# 06 - Trajectory Optimization

## Scenario: Few-Shot Trajectory Injection

If your agent is struggling to figure out how to chain tools together (e.g., it checks the server, then asks the user for help, instead of immediately checking the logs), you don't always need a better model. 

You can inject **Golden Trajectories** into the prompt. A trajectory is simply a mocked conversation history showing exactly how you *want* the agent to behave.

In this notebook, we will demonstrate how to inject a fake conversation history into the `messages` array.

In [1]:
import os
from openai import OpenAI

# 1. Attempt to use the real API
if os.environ.get("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
else:
    # 2. Fallback to our local mock for students without keys
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    import sys
    import os
    sys.path.append(os.path.abspath("../../.."))
    from awsome_agents.mock_openai import MockOpenAI
    client = MockOpenAI()

# 3. Optional: Local LLMs
# If you prefer to use a local model like Llama 3 instead of the mock:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# A normal system prompt
base_messages = [
    {"role": "system", "content": "You are an SRE agent. If a server is failing, check the logs immediately."}
]


## 1. Building the Golden Trajectory

We construct a fake user request, a fake tool call by the assistant, a fake tool observation, and the final response. By putting this *before* the actual user request, the LLM learns the exact pattern via few-shot prompting.

In [2]:
# This is the "Golden Trajectory" we want the agent to mimic
golden_trajectory = [
    # Fake User Request
    {"role": "user", "content": "The US-East checkout is down."},
    
    # Fake Assistant Tool Call
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": "call_golden1",
                "type": "function",
                "function": {"name": "check_server_health", "arguments": '{"region": "us-east"}'}
            }
        ]
    },
    
    # Fake Tool Result
    {
        "role": "tool",
        "tool_call_id": "call_golden1",
        "name": "check_server_health",
        "content": '{"status": "failing"}'
    },
    
    # Fake Assistant NEXT Action (The crucial step we want it to learn!)
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": "call_golden2",
                "type": "function",
                "function": {"name": "fetch_logs", "arguments": '{"region": "us-east"}'}
            }
        ]
    },
    
    # Fake Tool Result 2
    {
        "role": "tool",
        "tool_call_id": "call_golden2",
        "name": "fetch_logs",
        "content": 'Error: Redis timeout.'
    },
    
    # Fake Final Response
    {"role": "assistant", "content": "The US-East server is failing due to a Redis timeout. I am escalating to engineering."}
]

# When the real user asks a question, we append the golden trajectory first
real_user_request = {"role": "user", "content": "The EU-West checkout is throwing 500s."}

final_messages = base_messages + golden_trajectory + [real_user_request]

print(f"Total messages sent to LLM: {len(final_messages)}")
print("The LLM will now perfectly mimic the pattern of [Check Health -> Fetch Logs -> Escalate].")


Total messages sent to LLM: 8
The LLM will now perfectly mimic the pattern of [Check Health -> Fetch Logs -> Escalate].


## Checkpoint

**1. What is a 'Trajectory' in the context of AI Agents?**
- A) The physical location of the server.
- B) The sequence of Observations, Thoughts, and Actions (tool calls) taken by the agent to solve a problem.
- C) The memory usage of the python script.
- D) The learning rate of the model.
